# Notebook 06 — Popularidad (Modelo Baseline)

## ¿Qué vamos a construir?

Un **modelo de popularidad**: el recomendador más simple que existe. En vez de
personalizar recomendaciones para cada usuario, este modelo siempre sugiere
los productos que **más se vendieron en total**, sin importar quién sea el
usuario.

### ¿Por qué sirve esto?

Se lo llama **baseline** ("punto de partida") porque es el modelo más
sencillo posible, y sirve como piso de comparación: cualquier modelo más
sofisticado que construyamos después debería recomendar mejor que "siempre
sugerir lo más vendido".

También resuelve un problema muy concreto llamado **cold-start** ("arranque
en frío"): cuando un usuario es nuevo y todavía no compró nada, no tenemos
ningún historial suyo para personalizarle nada. En ese caso, mostrarle los
productos más populares (por ejemplo, bananas, que casi todo el mundo compra)
es una apuesta razonable.

### Lo que vamos a hacer, paso a paso

1. Cargar los datos crudos de Instacart.
2. Contar cuántas veces se compró cada producto.
3. Ordenar esos conteos de mayor a menor y armar un ranking (posición 1 = el
   más vendido).
4. Empaquetar ese ranking en una clase de Python reutilizable.
5. Guardar el modelo en disco con `joblib` para poder usarlo después sin
   tener que recalcular todo.
6. Recargarlo y verificar que funciona.

Esta notebook es **autónoma**: carga sus propios datos crudos y no depende
de que se haya ejecutado ninguna otra notebook antes.

## Paso 1 — Setup e imports

Antes de hacer nada necesitamos importar las herramientas que vamos a usar:

- **`pandas`**: la librería que usamos para trabajar con tablas de datos
  (los `DataFrame`), muy parecidas a una hoja de cálculo pero manejadas con
  código.
- **`joblib`**: una librería para *serializar* objetos de Python, es decir,
  guardarlos en un archivo tal cual están en memoria (con todos sus datos y
  su comportamiento) para poder cargarlos después sin tener que reconstruirlos
  desde cero.
- **`pathlib.Path`**: para armar rutas de archivos (`../data/...`) de forma
  que funcionen igual en Windows, Mac o Linux.
- **`sys`**: lo usamos solo para poder importar nuestra propia clase
  `PopularityRecommender`, que vive en la carpeta `src/` del proyecto (un
  nivel arriba de `notebooks/`).

También fijamos una semilla (`RANDOM_STATE`) por convención del proyecto,
aunque en este notebook en particular no hay ningún paso aleatorio (contar y
ordenar productos siempre da el mismo resultado).

In [1]:
import sys
import pandas as pd
import joblib
from pathlib import Path

# ── Rutas ─────────────────────────────────────────────────────────────────
# Esta notebook se ejecuta desde la carpeta notebooks/, por eso los paths
# relativos empiezan con '..' para "subir" a la raíz del proyecto.
DATA_RAW = Path('../data/raw/instacart')
MODELS   = Path('../models')

# Agregamos la carpeta src/ al "path" de Python para poder importar nuestra
# clase PopularityRecommender como si fuera una librería más.
_src_path = str(Path('../src').resolve())
if _src_path not in sys.path:
    sys.path.insert(0, _src_path)
from popularity_recommender import PopularityRecommender

RANDOM_STATE = 42  # fijo por convención del proyecto (acá no se usa aleatoriedad)

print('Setup completo.')
print(f'  DATA_RAW : {DATA_RAW.resolve()}')
print(f'  MODELS   : {MODELS.resolve()}')

Setup completo.
  DATA_RAW : C:\henry\ProyectoFinal-DataScience-Henry\data\raw\instacart
  MODELS   : C:\henry\ProyectoFinal-DataScience-Henry\models


## Paso 2 — Cargar los datos crudos

Instacart nos da los datos repartidos en varios archivos CSV. Para este
modelo necesitamos tres:

| Archivo | ¿Qué contiene? |
|---|---|
| `orders.csv` | Un registro por cada pedido: qué usuario lo hizo, en qué conjunto está (`prior`, `train` o `test`), etc. |
| `order_products__prior.csv` | Un registro por cada producto dentro de cada pedido "prior" (qué se compró en cada pedido). |
| `products.csv` | El catálogo: el nombre de cada producto según su `product_id`. |

### ¿Por qué usamos solo los pedidos `prior`?

Instacart divide los pedidos de cada usuario en tres grupos (`eval_set`):

- **`prior`**: el historial de compras "pasado" de cada usuario. Es el único
  conjunto que vamos a usar para calcular popularidad.
- **`train`** y **`test`**: se reservan para *evaluar* modelos (predecir el
  próximo pedido). Si los usáramos acá para calcular qué es popular,
  estaríamos "haciendo trampa" al momento de evaluar más adelante — a esto se
  le llama **data leakage** (fuga de datos): usar información que en un
  escenario real todavía no tendríamos disponible.

Cargamos solo las columnas que necesitamos (no todo el archivo) para que sea
más rápido y use menos memoria — `order_products__prior.csv` tiene más de 32
millones de filas.

In [2]:
# orders.csv: nos interesa el id de pedido, el usuario y a qué conjunto pertenece
orders = pd.read_csv(
    DATA_RAW / 'orders.csv',
    usecols=['order_id', 'user_id', 'eval_set'],
    dtype={'order_id': 'int32', 'user_id': 'int32'},
)
orders_prior = orders[orders['eval_set'] == 'prior'].copy()

# order_products__prior.csv: qué producto se compró en cada pedido "prior"
order_products_prior = pd.read_csv(
    DATA_RAW / 'order_products__prior.csv',
    usecols=['order_id', 'product_id'],
    dtype={'order_id': 'int32', 'product_id': 'int32'},
)

# products.csv: el catálogo con el nombre de cada producto
products = pd.read_csv(
    DATA_RAW / 'products.csv',
    usecols=['product_id', 'product_name'],
    dtype={'product_id': 'int32'},
)

print(f'orders_prior          : {orders_prior.shape[0]:,} pedidos, {orders_prior["user_id"].nunique():,} usuarios')
print(f'order_products_prior  : {order_products_prior.shape[0]:,} filas (una por producto comprado)')
print(f'products               : {products.shape[0]:,} productos en el catálogo')

orders_prior          : 3,214,874 pedidos, 206,209 usuarios
order_products_prior  : 32,434,489 filas (una por producto comprado)
products               : 49,688 productos en el catálogo


## Paso 3 — Exploración rápida

Antes de calcular nada, miramos cómo son realmente las tablas que cargamos.
Esto ayuda a no asumir cosas incorrectas sobre los datos.

- `.head(3)` muestra las primeras 3 filas de cada tabla, para ver sus
  columnas y el tipo de valores que tienen.
- `product_id` es el identificador único de cada producto (un número), y
  `product_name` es el nombre legible ("Banana", "Organic Avocado", etc.).

In [3]:
print('=== orders_prior (primeras filas) ===')
display(orders_prior.head(3))

print('\n=== order_products_prior (primeras filas) ===')
display(order_products_prior.head(3))

print('\n=== products (primeras filas) ===')
display(products.head(3))

print(f'\nProductos únicos en el catálogo : {products["product_id"].nunique():,}')
print(f'Productos únicos comprados      : {order_products_prior["product_id"].nunique():,}')

=== orders_prior (primeras filas) ===


,order_id,user_id,eval_set
0,2539329,1,prior
1,2398795,1,prior
2,473747,1,prior



=== order_products_prior (primeras filas) ===


,order_id,product_id
0,2,33120
1,2,28985
2,2,9327



=== products (primeras filas) ===


,product_id,product_name
0,1,Chocolate Sandwich Cookies
1,2,All-Seasons Salt
2,3,Robust Golden Unsweetened Oolong Tea



Productos únicos en el catálogo : 49,688
Productos únicos comprados      : 49,677


## Paso 4 — Contar cuántas veces se compró cada producto

Acá está el corazón del modelo de popularidad: contar, para cada
`product_id`, en cuántas filas de `order_products_prior` aparece. Cada fila
de esa tabla representa "este producto se agregó a este pedido", así que
contar filas por producto es literalmente contar cuántas veces se compró.

### ¿Qué es `groupby`?

`groupby('product_id')` es como armar un montón de pilas de papeles: agarra
todas las filas y las agrupa según su `product_id`, formando una "pila" por
cada producto distinto. Después, `.size()` cuenta cuántas filas (cuántos
"papeles") hay en cada pila. El resultado es: para cada producto, cuántas
veces aparece en total en los pedidos.

In [4]:
# Para cada product_id, contamos en cuántas filas aparece (= cuántas veces se compró)
purchase_counts = order_products_prior.groupby('product_id').size()

print(f'Cantidad de productos distintos comprados: {len(purchase_counts):,}')
print(f'Producto comprado más veces               : {purchase_counts.max():,} veces')
print(f'Producto comprado menos veces              : {purchase_counts.min():,} vez/veces')
print(f'Promedio de compras por producto           : {purchase_counts.mean():.1f}')

Cantidad de productos distintos comprados: 49,677
Producto comprado más veces               : 472,565 veces
Producto comprado menos veces              : 1 vez/veces
Promedio de compras por producto           : 652.9


## Paso 5 — Armar el ranking

Ahora que tenemos el conteo de compras por producto, lo convertimos en un
ranking:

1. **Ordenamos** los conteos de mayor a menor (`sort_values(ascending=False)`):
   el producto más comprado va primero.
2. **Asignamos una posición** (`rank`) a cada uno: 1 para el más vendido, 2
   para el segundo más vendido, y así sucesivamente.
3. Hacemos un **`merge`** (unión) con la tabla `products` para agregarle el
   nombre real del producto. Un `merge` es como un "buscar y traer": por
   cada `product_id` en nuestro ranking, busca ese mismo `product_id` en la
   tabla `products` y trae su `product_name`. Es el mismo concepto que un
   `VLOOKUP` en Excel, o un `JOIN` en SQL.

In [5]:
# 1) Ordenar de mayor a menor cantidad de compras
ranking = (
    purchase_counts
    .sort_values(ascending=False)
    .reset_index(name='purchase_count')  # convierte el resultado agrupado en un DataFrame normal
)

# 2) Asignar la posición del ranking: 1 = más vendido
ranking['rank'] = range(1, len(ranking) + 1)

# 3) Traer el nombre del producto desde la tabla 'products'
ranking = ranking.merge(products, on='product_id', how='left')

# Reordenamos las columnas para que sea más fácil de leer
ranking = ranking[['rank', 'product_id', 'product_name', 'purchase_count']]

print(f'Ranking armado con {len(ranking):,} productos.')
ranking.head(10)

Ranking armado con 49,677 productos.


,rank,product_id,product_name,purchase_count
0,1,24852,Banana,472565
1,2,13176,Bag of Organic Bananas,379450
2,3,21137,Organic Strawberries,264683
3,4,21903,Organic Baby Spinach,241921
4,5,47209,Organic Hass Avocado,213584
5,6,47766,Organic Avocado,176815
6,7,47626,Large Lemon,152657
7,8,16797,Strawberries,142951
8,9,26209,Limes,140627
9,10,27845,Organic Whole Milk,137905


## Paso 6 — Vista previa del Top 10

Antes de empaquetar todo en una clase, chequeamos "a mano" que el top 10 se
vea razonable. En Instacart es conocido que los productos frescos (frutas y
verduras de compra habitual) dominan el ranking, así que esperamos ver cosas
como bananas, paltas (avocado) o espinaca.

In [6]:
top_10 = ranking.head(10)

for _, row in top_10.iterrows():
    print(f"producto: {row['product_name']}, rank: {row['rank']}°  (comprado {row['purchase_count']:,} veces)")

producto: Banana, rank: 1°  (comprado 472,565 veces)
producto: Bag of Organic Bananas, rank: 2°  (comprado 379,450 veces)
producto: Organic Strawberries, rank: 3°  (comprado 264,683 veces)
producto: Organic Baby Spinach, rank: 4°  (comprado 241,921 veces)
producto: Organic Hass Avocado, rank: 5°  (comprado 213,584 veces)
producto: Organic Avocado, rank: 6°  (comprado 176,815 veces)
producto: Large Lemon, rank: 7°  (comprado 152,657 veces)
producto: Strawberries, rank: 8°  (comprado 142,951 veces)
producto: Limes, rank: 9°  (comprado 140,627 veces)
producto: Organic Whole Milk, rank: 10°  (comprado 137,905 veces)


## Paso 7 — Empaquetar el ranking en una clase reutilizable

Podríamos quedarnos con el `DataFrame` `ranking` tal cual, pero conviene
envolverlo en una **clase** de Python: `PopularityRecommender`, definida en
[`src/popularity_recommender.py`](../src/popularity_recommender.py).

### ¿Por qué una clase y no solo un DataFrame?

Una clase junta **datos** (el ranking) y **comportamiento** (el método
`recommend(n)`, que sabe cómo transformar ese ranking en una respuesta lista
para usar) en un solo objeto. Así, quien use el modelo más adelante no
necesita saber que por dentro hay un DataFrame ni cómo está armado el
ranking — solo necesita llamar `modelo.recommend(10)` y listo. Esto es lo
que en programación se llama **encapsular**: esconder los detalles internos
detrás de una interfaz simple.

La clase vive en su propio archivo `.py` (no acá en la notebook) porque más
adelante vamos a guardarla en disco con `joblib`, y para poder volver a
cargarla en *cualquier* script Python (no solo en esta notebook), la
definición de la clase tiene que existir como un módulo importable.

In [7]:
# Instanciamos el modelo con el ranking que acabamos de calcular
modelo_popularidad = PopularityRecommender(ranking_df=ranking, metadata={'n_productos': len(ranking)})

print(modelo_popularidad)

# Probamos el método recommend(): debería devolver una lista de diccionarios
recomendaciones = modelo_popularidad.recommend(10)

print('\nRecomendaciones para un usuario sin historial (top 10):')
for r in recomendaciones:
    print(f"producto: {r['producto']}, rank: {r['rank']}°")

PopularityRecommender(n_productos=49677)

Recomendaciones para un usuario sin historial (top 10):
producto: Banana, rank: 1°
producto: Bag of Organic Bananas, rank: 2°
producto: Organic Strawberries, rank: 3°
producto: Organic Baby Spinach, rank: 4°
producto: Organic Hass Avocado, rank: 5°
producto: Organic Avocado, rank: 6°
producto: Large Lemon, rank: 7°
producto: Strawberries, rank: 8°
producto: Limes, rank: 9°
producto: Organic Whole Milk, rank: 10°


Chequeamos que la salida cumple lo que esperamos: una lista de 10
diccionarios, cada uno con las claves `producto` y `rank`, ordenados del 1°
al 10°.

In [8]:
assert isinstance(recomendaciones, list), 'debe devolver una lista'
assert len(recomendaciones) == 10, f'debe tener 10 elementos, tiene {len(recomendaciones)}'
assert all(isinstance(r, dict) for r in recomendaciones), 'cada elemento debe ser un diccionario'
assert all({'producto', 'rank'} <= r.keys() for r in recomendaciones), 'faltan claves producto/rank'
assert [r['rank'] for r in recomendaciones] == list(range(1, 11)), 'el rank debe ir de 1 a 10 en orden'

print('✓ La salida cumple el formato esperado.')

✓ La salida cumple el formato esperado.


## Paso 8 — Guardar el modelo con `joblib`

### ¿Qué significa "serializar"?

Ahora mismo, `modelo_popularidad` es un objeto que solo existe en la memoria
de esta notebook: si cerramos el notebook, se pierde. **Serializar** un
objeto significa convertirlo en un archivo que se puede guardar en disco y
volver a cargar después, reconstruyendo el objeto exactamente como estaba
(con todos sus datos).

### ¿Por qué `joblib` y no, por ejemplo, guardar el ranking como CSV?

Un CSV solo puede guardar datos en forma de tabla (filas y columnas). Acá
queremos guardar el **objeto completo** `PopularityRecommender` — que
incluye el DataFrame, pero también sabe "cómo comportarse" (el método
`recommend`). `joblib` puede serializar objetos de Python en general
(no solo tablas), y además está optimizado para guardar de forma eficiente
objetos que contienen arrays y DataFrames grandes — por eso es el estándar
en proyectos de machine learning, y el que se usa en el resto de los
modelos de este proyecto.

In [9]:
MODELS.mkdir(exist_ok=True)

model_path = MODELS / 'popularity_model.joblib'
joblib.dump(modelo_popularidad, model_path)

print(f'✓ Modelo guardado en: {model_path.resolve()}')

✓ Modelo guardado en: C:\henry\ProyectoFinal-DataScience-Henry\models\popularity_model.joblib


## Paso 9 — Recargar y verificar

Como última prueba, cargamos el modelo desde el archivo que acabamos de
escribir (como si fuéramos otro script distinto que solo tiene el archivo
`.joblib`, sin haber calculado nada) y confirmamos que produce exactamente
el mismo top 10. Esto nos da confianza de que el modelo guardado realmente
sirve para usarse después, sin depender de esta notebook.

In [10]:
modelo_cargado = PopularityRecommender.load(model_path)

recomendaciones_recargadas = modelo_cargado.recommend(10)

print('Recomendaciones del modelo recargado desde disco:')
for r in recomendaciones_recargadas:
    print(f"producto: {r['producto']}, rank: {r['rank']}°")

assert recomendaciones_recargadas == recomendaciones, 'el modelo recargado debería dar la misma salida'
print('\n✓ El modelo recargado da exactamente el mismo resultado que el original.')

Recomendaciones del modelo recargado desde disco:
producto: Banana, rank: 1°
producto: Bag of Organic Bananas, rank: 2°
producto: Organic Strawberries, rank: 3°
producto: Organic Baby Spinach, rank: 4°
producto: Organic Hass Avocado, rank: 5°
producto: Organic Avocado, rank: 6°
producto: Large Lemon, rank: 7°
producto: Strawberries, rank: 8°
producto: Limes, rank: 9°
producto: Organic Whole Milk, rank: 10°

✓ El modelo recargado da exactamente el mismo resultado que el original.


## Cierre

Construimos un modelo de **popularidad global**: cuenta cuántas veces se
compró cada producto, arma un ranking, y expone un método `recommend(n)`
que devuelve el top-n en formato `{'producto': ..., 'rank': ...}`. Lo
guardamos con `joblib` en `models/popularity_model.joblib`, listo para
usarse desde cualquier otro script sin volver a procesar los datos crudos.

Este modelo es el **piso** (fallback) para usuarios sin historial de
compras: cualquier modelo más sofisticado que se construya en las próximas
notebooks (basado en historial del usuario, en reglas de asociación entre
productos, etc.) debería recomendar mejor que esto para justificar su
complejidad extra.